In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import hashlib

# Resolve the project root whether Jupyter starts in the repository root
# or inside the notebooks directory.
current_dir = Path.cwd()
if (current_dir / 'data' / 'results.csv').exists():
    project_root = current_dir
elif (current_dir.parent / 'data' / 'results.csv').exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError('Could not locate data/results.csv.')
results_path = project_root / 'data' / 'results.csv'
outputs_dir = project_root / 'outputs'
outputs_dir.mkdir(exist_ok=True)
df_matches = pd.read_csv(results_path)
df_matches['date'] = pd.to_datetime(df_matches['date'], errors='coerce')
print('Project root:', project_root)
print('Dataset path:', results_path)
print('Dataset Shape:', df_matches.shape)
display(df_matches.head())


In [ ]:
# Validate completed matches and create deterministic match identifiers.
required_columns = [
    'date', 'home_team', 'away_team', 'home_score', 'away_score',
    'tournament', 'city', 'country', 'neutral'
]
missing_columns = [c for c in required_columns if c not in df_matches.columns]
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

df_matches['home_score'] = pd.to_numeric(df_matches['home_score'], errors='coerce')
df_matches['away_score'] = pd.to_numeric(df_matches['away_score'], errors='coerce')
df_matches = df_matches.dropna(subset=['date', 'home_score', 'away_score']).copy()
df_matches = df_matches.drop_duplicates(subset=required_columns, keep='first').copy()

def make_match_id(row):
    values = []
    for column in required_columns:
        value = row[column]
        if column == 'date':
            value = value.strftime('%Y-%m-%d')
        elif isinstance(value, float) and value.is_integer():
            value = int(value)
        values.append(str(value).strip())
    return 'match_' + hashlib.sha1('|'.join(values).encode('utf-8')).hexdigest()[:16]
df_matches['match_id'] = df_matches.apply(make_match_id, axis=1)
if not df_matches['match_id'].is_unique:
    raise ValueError('Deterministic match_id is not unique.')

df_clean = df_matches[df_matches['date'] >= '2000-01-01'].copy()
print('Validated completed matches:', len(df_matches))
print('Total matches after 2000:', len(df_clean))
print('Top 10 Tournament Types:')
print(df_clean['tournament'].value_counts().head(10))


In [ ]:
from collections import deque

df_home = df_clean[['match_id','date','home_team','home_score','away_score']].copy()
df_home['side']='home'
df_home=df_home.rename(columns={'home_team':'team','home_score':'goals_for','away_score':'goals_against'})
df_away = df_clean[['match_id','date','away_team','away_score','home_score']].copy()
df_away['side']='away'
df_away=df_away.rename(columns={'away_team':'team','away_score':'goals_for','home_score':'goals_against'})
df_team_matches=pd.concat([df_home,df_away],ignore_index=True).sort_values(['team','date','match_id','side']).reset_index(drop=True)
expected_team_rows=2*len(df_clean)
if len(df_team_matches)!=expected_team_rows: raise ValueError('Unexpected team-perspective row count.')

window_size=5
df_team_matches['form_goals_for']=(df_team_matches.groupby('team')['goals_for'].transform(lambda s:s.shift(1).rolling(window_size,min_periods=1).mean()))
df_team_matches['form_goals_against']=(df_team_matches.groupby('team')['goals_against'].transform(lambda s:s.shift(1).rolling(window_size,min_periods=1).mean()))
df_team_matches['form_goals_for']=df_team_matches.groupby(['team','date'])['form_goals_for'].transform('first').fillna(0.0)
df_team_matches['form_goals_against']=df_team_matches.groupby(['team','date'])['form_goals_against'].transform('first').fillna(0.0)
same_day_rows=df_team_matches[df_team_matches.duplicated(['team','date'],keep=False)]
same_day_form_counts=same_day_rows.groupby(['team','date'])[['form_goals_for','form_goals_against']].nunique()
if not same_day_form_counts.empty and (same_day_form_counts>1).any().any(): raise ValueError('Same-day form leakage detected.')
print('Team-perspective rows:',len(df_team_matches))
print('Expected team-perspective rows:',expected_team_rows)
print('Same-day team/date groups handled:',len(same_day_form_counts))
display(df_team_matches[df_team_matches['team']=='Argentina'].tail(10))


In [ ]:
home_form=df_team_matches[df_team_matches['side'].eq('home')][['match_id','form_goals_for','form_goals_against']].rename(columns={'form_goals_for':'home_form_goals_for','form_goals_against':'home_form_goals_against'})
away_form=df_team_matches[df_team_matches['side'].eq('away')][['match_id','form_goals_for','form_goals_against']].rename(columns={'form_goals_for':'away_form_goals_for','form_goals_against':'away_form_goals_against'})
df_model=(df_clean.merge(home_form,on='match_id',how='left',validate='one_to_one').merge(away_form,on='match_id',how='left',validate='one_to_one'))
form_columns=['home_form_goals_for','home_form_goals_against','away_form_goals_for','away_form_goals_against']
if df_model[form_columns].isna().any().any(): raise ValueError('Missing rolling-form values after match_id merge.')
if len(df_model)!=len(df_clean) or not df_model['match_id'].is_unique: raise ValueError('Rolling-form merge changed one-row-per-match structure.')
conditions=[df_model['home_score']>df_model['away_score'],df_model['home_score']==df_model['away_score'],df_model['home_score']<df_model['away_score']]
df_model['target']=np.select(conditions,[2,1,0],default=np.nan)
if df_model['target'].isna().any(): raise ValueError('Target encoding produced missing values.')
rolling_summary=pd.DataFrame([('modern_matches',len(df_clean)),('team_perspective_rows',len(df_team_matches)),('same_day_team_date_groups',len(same_day_form_counts)),('final_model_rows',len(df_model)),('unique_model_match_ids',df_model['match_id'].nunique()),('duplicate_model_match_ids',int(df_model['match_id'].duplicated().sum())),('rows_added_or_lost',len(df_model)-len(df_clean)),('missing_form_values',int(df_model[form_columns].isna().sum().sum()))],columns=['metric','value'])
rolling_summary.to_csv(outputs_dir/'rolling_form_validation.csv',index=False)
print('Final Feature Matrix Shape:',df_model.shape)
display(rolling_summary)


In [ ]:
import unicodedata

def normalise_tournament_name(tournament_name):
    # Normalise case and accents so 'Copa América' and 'Copa America'
    # are treated consistently.
    text = str(tournament_name).strip()
    text = unicodedata.normalize('NFKD', text).encode('ascii','ignore').decode('ascii')
    return text.casefold()

def get_tournament_weight(tournament_name):
    name = normalise_tournament_name(tournament_name)
    if 'fifa world cup' in name and 'qualification' not in name:
        return 1.0
    major_finals = ('confederations cup', 'copa america', 'euro')
    if any(term in name for term in major_finals) and 'qualification' not in name:
        return 0.8
    if 'qualification' in name:
        return 0.6
    if 'nations league' in name:
        return 0.5
    if 'friendly' in name:
        return 0.25
    return 0.4

df_clean['match_weight']=df_clean['tournament'].apply(get_tournament_weight)
df_clean['is_neutral']=df_clean['neutral'].astype(int)

# Add context features to the existing Commit 4 model table using match_id.
# This keeps the one-row-per-match structure and avoids date/team joins.
context_features = df_clean[['match_id', 'match_weight', 'is_neutral']]
df_model = df_model.merge(context_features, on='match_id', how='left', validate='one_to_one')
if df_model[['match_weight', 'is_neutral']].isna().any().any():
    raise ValueError('Missing tournament context after match_id merge.')

features=['home_form_goals_for','home_form_goals_against','away_form_goals_for','away_form_goals_against','match_weight','is_neutral']
X=df_model[features]
y=df_model['target']

weight_validation=(df_clean.groupby(['tournament','match_weight']).size().reset_index(name='match_count').sort_values(['match_weight','tournament'],ascending=[False,True]))
weight_validation.to_csv(outputs_dir/'tournament_weight_validation.csv',index=False)
copa_count=int((df_clean['tournament']=='Copa América').sum())
copa_weight_counts=df_clean.loc[df_clean['tournament']=='Copa América','match_weight'].value_counts().to_dict()
if copa_count != 248: raise ValueError(f'Expected 248 modern-era Copa América matches, found {copa_count}.')
if copa_weight_counts.get(0.8,0) != 248: raise ValueError(f'Copa América weight validation failed: {copa_weight_counts}')
if copa_weight_counts.get(0.4,0) != 0: raise ValueError('Copa América still contains generic 0.4 assignments.')
print('Copa América matches:',copa_count)
print('Copa América assigned weights:',copa_weight_counts)
print('Weight validation written to:',outputs_dir/'tournament_weight_validation.csv')
display(weight_validation[weight_validation['tournament'].isin(['Copa América','Copa América qualification','FIFA World Cup','FIFA World Cup qualification','UEFA Euro','UEFA Euro qualification','Confederations Cup','Friendly','UEFA Nations League'])])


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# 1. Define our Feature Matrix (X) and Target Vector (y)
# We isolate only the predictive numeric columns, dropping metadata like dates and names
features = [
    'home_form_goals_for', 'home_form_goals_against',
    'away_form_goals_for', 'away_form_goals_against',
    'match_weight', 'is_neutral'
]

X = df_model[features]
y = df_model['target']

# 2. Split the pitch: 80% for training, 20% for testing our predictions
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Standardize the data so the SVM hyperplanes aren't distorted
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Initialize and train the model WITH balanced class weights
print("Retraining the SVM with balanced weights...")
svm_model_balanced = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
svm_model_balanced.fit(X_train_scaled, y_train)

# 5. Evaluate the newly balanced model
y_pred_balanced = svm_model_balanced.predict(X_test_scaled)
print("\n--- Balanced Model Evaluation ---")
print(classification_report(y_test, y_pred_balanced, target_names=['Away Win (0)', 'Draw (1)', 'Home Win (2)']))

In [ ]:
# 1. Define standard Elo mathematical functions
def get_expected_score(rating_a, rating_b):
    # Calculates the probability of Team A winning based on Elo difference
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def update_elo(rating, expected, actual, k=30):
    # k is the weight of the match.
    return rating + k * (actual - expected)

# 2. Initialize a dictionary to track everyone's live rating (Baseline is 1500)
current_elo = {}

home_elos = []
away_elos = []

# 3. Loop through history chronologically to calculate pre-match Elo for every game
print("Calculating historical Elo ratings...")
for index, row in df_clean.sort_values('date').iterrows():
    home = row['home_team']
    away = row['away_team']

    # If a team is new to the dataset, give them the baseline 1500 rating
    if home not in current_elo: current_elo[home] = 1500
    if away not in current_elo: current_elo[away] = 1500

    # Store the pre-match ratings to use as features later
    home_elos.append(current_elo[home])
    away_elos.append(current_elo[away])

    # Determine the actual outcome for the Elo math (1 = Home Win, 0.5 = Draw, 0 = Away Win)
    if row['home_score'] > row['away_score']:
        actual_home, actual_away = 1.0, 0.0
    elif row['home_score'] < row['away_score']:
        actual_home, actual_away = 0.0, 1.0
    else:
        actual_home, actual_away = 0.5, 0.5

    # Calculate expected outcomes
    expected_home = get_expected_score(current_elo[home], current_elo[away])
    expected_away = get_expected_score(current_elo[away], current_elo[home])

    # Update their live ratings using the match weight we created earlier
    k_adjusted = 30 * row['match_weight']
    current_elo[home] = update_elo(current_elo[home], expected_home, actual_home, k_adjusted)
    current_elo[away] = update_elo(current_elo[away], expected_away, actual_away, k_adjusted)

# 4. Attach these shiny new features to our clean dataset
df_clean_sorted = df_clean.sort_values('date').copy()
df_clean_sorted['home_elo'] = home_elos
df_clean_sorted['away_elo'] = away_elos

# Let's see who the top 5 teams are at the end of our dataset
print("\nTop 5 Teams by Final Elo Rating:")
top_teams = sorted(current_elo.items(), key=lambda x: x[1], reverse=True)[:5]
for team, rating in top_teams:
    print(f"{team}: {rating:.0f}")

In [ ]:
# 1. Merge the new Elo ratings into our existing model dataframe
df_model = pd.merge(
    df_model,
    df_clean_sorted[['date', 'home_team', 'away_team', 'home_elo', 'away_elo']],
    on=['date', 'home_team', 'away_team'],
    how='left'
)

# Clean up any potential NaNs from the merge
df_model = df_model.dropna()

# 2. Define our Upgraded Feature Matrix (X)
features_upgraded = [
    'home_elo', 'away_elo', # <-- The new heavy hitters
    'home_form_goals_for', 'home_form_goals_against',
    'away_form_goals_for', 'away_form_goals_against',
    'match_weight', 'is_neutral'
]

X_upgraded = df_model[features_upgraded]
y_upgraded = df_model['target']

# 3. Split and Scale the new pitch
X_train_up, X_test_up, y_train_up, y_test_up = train_test_split(X_upgraded, y_upgraded, test_size=0.2, random_state=42)

scaler_upgraded = StandardScaler()
X_train_scaled_up = scaler_upgraded.fit_transform(X_train_up)
X_test_scaled_up = scaler_upgraded.transform(X_test_up)

# 4. Retrain the SVM Engine
print("Retraining the SVM with Historical Elo Ratings...")
svm_model_final = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
svm_model_final.fit(X_train_scaled_up, y_train_up)

# 5. Evaluate the Ultimate Model
y_pred_final = svm_model_final.predict(X_test_scaled_up)
print("\n--- Final Model Evaluation (Elo + Form) ---")
print(classification_report(y_test_up, y_pred_final, target_names=['Away Win (0)', 'Draw (1)', 'Home Win (2)']))

In [ ]:
import numpy as np
from collections import Counter

# 1. The Match Prediction Engine
def get_match_probabilities(team_home, team_away):
    # In a real app, you would dynamically pull their live Elo and Form here.
    # For this simulation, we will construct a dummy feature vector representing a tight match.
    # Features: [home_elo, away_elo, home_gf, home_ga, away_gf, away_ga, weight, neutral]
    # Let's pretend they are evenly matched (Elo 1800) on neutral ground (1.0 weight, 1 neutral)
    dummy_features = [[1800, 1750, 2.0, 0.5, 1.8, 0.8, 1.0, 1]]

    # Scale and predict
    scaled_features = scaler_upgraded.transform(dummy_features)
    probs = svm_model_final.predict_proba(scaled_features)[0]

    # probs output is [P(Away Win), P(Draw), P(Home Win)]
    return probs[0], probs[1], probs[2]

# 2. The Single Match Simulator
def simulate_knockout_match(team_a, team_b):
    p_away, p_draw, p_home = get_match_probabilities(team_a, team_b)

    # In knockout football, there are no draws. If the SVM predicts a draw,
    # we simulate extra time/penalties by essentially tossing a coin (50/50).
    outcomes = [team_b, 'Draw', team_a]
    result = np.random.choice(outcomes, p=[p_away, p_draw, p_home])

    if result == 'Draw':
        return np.random.choice([team_a, team_b])
    return result

# 3. The Monte Carlo Bracket Simulator
def run_monte_carlo_tournament(iterations=10000):
    champions = []

    print(f"Running Monte Carlo Simulation ({iterations} universes)...")

    for _ in range(iterations):
        # Semi-Finals
        finalist_1 = simulate_knockout_match("Spain", "Brazil")
        finalist_2 = simulate_knockout_match("France", "Argentina")

        # The Final
        winner = simulate_knockout_match(finalist_1, finalist_2)
        champions.append(winner)

    return Counter(champions)

# 4. Execute the Simulation
results = run_monte_carlo_tournament(10000)

print("\n--- World Cup Monte Carlo Results (10,000 Simulations) ---")
total = sum(results.values())
for team, wins in results.most_common():
    win_percentage = (wins / total) * 100
    print(f"{team}: {wins} tournament wins ({win_percentage:.2f}%)")

Chapter 1

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np

# 1. The Correlation Matrix
# We analyze X_upgraded to see how our engineered features interact
plt.figure(figsize=(10, 8))
correlation_matrix = X_upgraded.corr()

# Generate a clean heatmap using Seaborn
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Matrix: Pre-Match Variables")
plt.tight_layout()
plt.show()

# 2. Principal Component Analysis (PCA)
# We apply PCA to our SCALED training data to see the mathematical variance
pca = PCA()
pca.fit(X_train_scaled_up)

# Calculate the cumulative explained variance
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

# Plot the PCA curve
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='--', color='b')
plt.axhline(y=0.90, color='r', linestyle='-') # The 90% variance threshold
plt.text(1.5, 0.91, '90% Variance Threshold', color='red')

plt.title("PCA: Cumulative Explained Variance")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Variance")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3. Print the hard numbers for your thesis text
print("--- PCA Breakdown ---")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"Principal Component {i+1}: {var*100:.2f}% of variance explained")

Chapter 2

In [ ]:
from sklearn.neural_network import MLPClassifier

# 1. Initialize the Neural Network Architecture
# We'll use two hidden layers (16 neurons, then 8 neurons) with ReLU activation.
# max_iter is set to 1000 to ensure the network has time to converge.
print("Training the Deep Learning Engine (MLP)...")
mlp_model = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    activation='relu',
    solver='adam',
    max_iter=1000,
    random_state=42
)

# 2. Fit the model to our scaled Elo + Form data
mlp_model.fit(X_train_scaled_up, y_train_up)

# 3. Evaluate the Neural Network
y_pred_mlp = mlp_model.predict(X_test_scaled_up)
print("\n--- Neural Network (MLP) Evaluation ---")
print(classification_report(y_test_up, y_pred_mlp, target_names=['Away Win (0)', 'Draw (1)', 'Home Win (2)']))

# 4. Compare the baseline accuracy of the two architectures
svm_acc = svm_model_final.score(X_test_scaled_up, y_test_up)
mlp_acc = mlp_model.score(X_test_scaled_up, y_test_up)

print(f"\nModel Showdown - SVM Accuracy: {svm_acc*100:.2f}% | MLP Accuracy: {mlp_acc*100:.2f}%")